# 01 - Fixed V63 features and temporal inputs

Audit the fixed 49 features, reconcile source order and encoding, construct calendar grids, then reconstruct monthly and quarterly histories only after verified builders are supplied. Position 0 is newest; the current month ends at END_DT. Four quarterly positions group the same 12 monthly buckets. Rolling lookbacks may extend earlier than those buckets.

**Input contract:** 49 ordered V63 predictors, 23,151 snapshots from 12,447 patients and 1,345 positives; PATIENT_ID + END_DT and RESP remain unchanged. No feature selection, new sampling or population update runs here.

**Reconstruction gate:** notebook 01 still requires verified raw feature calculations, observation coverage and historical availability. Date helpers and encoding formulas do not establish reconstructed history; unsupported inputs stop the pipeline. Current-state sources are not historical as-of evidence.

**Evaluation limits:** frozen V63 caps/types were originally fitted using RESP, and their upstream population has not been checked against these held-out assignments. TRAIN-only scaling does not resolve that limitation. TEST was previously inspected. Monthly/quarterly and masking are hypotheses; this comparison does not isolate a causal masking effect or establish improved performance.

Use these four notebooks in order. The older build scripts and folder guide describe the previous experiment and must not regenerate this revision. Save sensitive artifacts only through the existing private warehouse path and clear executed outputs before sharing notebook code.


The supplied 49-name registration was reviewed separately from the authoritative warehouse list. Its final overrides replayed snapshot values for every feature; those overrides are omitted. AGE uses demographics at each cutoff. Twelve event drafts still lack reliable coverage/provenance and contain unresolved calculation details; the other 36 entries supply replay rather than historical calculations, including a conflicting ratio definition. Their sources and gaps are displayed below. Training remains blocked until every configured predictor has a supported historical builder.


In [ ]:
# Connection and unchanged experimental settings
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply the existing private sf_options connection on the approved Spark runtime.')
sf_options_dl_poc = dict(sf_options)
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'H002'
RUN_ID = 'M001'
import re
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use short uppercase identifiers; use matching IDs in all four notebooks.')
# New output names prevent collision with prior saved models; original inputs remain unchanged.
EXPERIMENT_PREFIX = PREFIX + '_BUSINESS49_TEMPORAL_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
REFERENCE_MODEL_TABLE = PREFIX + '_MODEL_RUN_001'
REFERENCE_NAMES = {'checkpoint.pt', 'training_summary.json', 'training_history.csv', 'training_history.png'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device='auto')
print('Fixed V63 business features; source cohort and RESP retained. Run MONTHLY then QUARTERLY.')



IMPLEMENTATION_SHA256 = 'b1d31cb7516eec376eab5768362a4b45f805692c15c3284bf1755e026c8f657f'


In [ ]:
# Embedded validation and warehouse helpers
"""Embedded notebook helpers: fixed business features, calendar grids and padding."""
import io
import json
import hashlib
import marshal
import numpy as np
import pandas as pd


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ordered_features():
    features, comparison = configured_features()
    require(len(features) == 49, 'V63 MODEL_TYPE.FEATURES must contain exactly 49 predictors.')
    return features, comparison


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def sequence_grid(metadata, representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    rows = []
    for r in metadata.itertuples():
        cutoff = pd.Timestamp(r.END_DT)
        month = cutoff.to_period('M')
        for step in range(12 // width):
            start = (month - width * step - (width - 1)).start_time.normalize()
            end = min(cutoff, (month - width * step).end_time.normalize())
            rows.append((r.PATIENT_ID, r.END_DT, step, start.strftime('%Y-%m-%d'),
                         end.strftime('%Y-%m-%d'), int(r.RESP)))
    grid = pd.DataFrame(rows, columns=['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'RESP'])
    require(not grid.duplicated(['PATIENT_ID', 'END_DT', 'TIME_STEP']).any(), 'Duplicate sequence keys.')
    require(grid.PERIOD_END.le(grid.END_DT).all(), 'Future period boundary.')
    return grid


def verify_periods(monthly, quarterly):
    keys = ['PATIENT_ID', 'END_DT']
    for frame, count in ((monthly, 12), (quarterly, 4)):
        require(not frame.duplicated(keys + ['TIME_STEP']).any(), 'Duplicate sequence row.')
        require(frame.groupby(keys).TIME_STEP.apply(lambda x: sorted(x) == list(range(count))).all(),
                'Missing or invalid sequence positions.')
    for q in range(4):
        m = monthly.loc[monthly.TIME_STEP.between(q * 3, q * 3 + 2)]
        bounds = m.groupby(keys).agg(PERIOD_START=('PERIOD_START', 'min'), PERIOD_END=('PERIOD_END', 'max'))
        actual = quarterly.loc[quarterly.TIME_STEP.eq(q)].set_index(keys)[['PERIOD_START', 'PERIOD_END']].sort_index()
        require(bounds.sort_index().equals(actual), 'Monthly and quarterly calendar periods differ.')


def reconcile_dictionary(features, dictionary):
    rows, assigned = [], set()
    for order, name in enumerate(features):
        exact = [r for r in dictionary if r['name_complete'] and r['name'] == name]
        candidates = exact or [r for r in dictionary if not r['name_complete'] and name.startswith(r['name'])]
        match = candidates[0] if len(candidates) == 1 else None
        if match:
            require(match['seq'] not in assigned, 'Dictionary entry matched multiple model features; confirm full names.')
            assigned.add(match['seq'])
        rows.append({'FEATURE_ORDER': order, 'FEATURE_NAME': name,
                     'DICTIONARY_SEQ': match['seq'] if match else None,
                     'MATCH': ('EXACT_NAME' if exact else 'UNIQUE_VISIBLE_PREFIX') if match else 'UNRESOLVED',
                     'FEATURE_TYPE': (match['type_label'] + ' (dictionary label; not a casting rule)') if match else 'UNRESOLVED',
                     'DEFAULT_STATUS': match['default_status'] if match else 'UNRESOLVED',
                     'NOTES': ((match.get('notes', '') + ('; clipped name matched by unique visible prefix' if not exact else ''))
                               if match else 'Confirm full feature name/definition against V63.')})
    unmatched = pd.DataFrame([r for r in dictionary if r['seq'] not in assigned])
    return pd.DataFrame(rows), unmatched


def reconstruction_audit(features, dictionary, rules, parameters):
    matched, extra = reconcile_dictionary(features, dictionary)
    require(not set(rules).difference(features), 'Historical rules include non-V63 features.')
    rows = []
    for row in matched.to_dict('records'):
        name = row['FEATURE_NAME']
        rule = rules.get(name)
        source, logic = feature_lineage_notes(name)
        parameter = parameters[row['FEATURE_ORDER']]
        require(parameter['FEATURES'] == name, 'Audit and fixed parameter order differ.')
        row['DICTIONARY_TYPE'] = row['FEATURE_TYPE']
        row['FEATURE_TYPE'] = parameter['VAR_TYP'] + ' (fixed V63 output encoding)'
        row['VALUE_P'] = parameter['VALUE_P']
        row['VALUE_TRANSFORMATION'] = parameter['TRANSFORM']
        row['SOURCE/CALCULATION'] = source
        row['HISTORICAL_RECONSTRUCTION_STATUS'] = row.pop('DEFAULT_STATUS')
        row['HISTORICAL_LOGIC'] = logic
        row['NOTES'] += '; Frozen VAR_TYP/VALUE_P applied once after raw reconstruction; patient coverage and source availability still require evidence.'
        if rule:
            status = rule.get('status')
            require(status in ('EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'), 'Invalid reconstruction status.')
            row['HISTORICAL_RECONSTRUCTION_STATUS'] = status
            row['SOURCE/CALCULATION'] = rule.get('source', '')
            row['HISTORICAL_LOGIC'] = rule.get('logic', '')
            row['NOTES'] += '; ' + rule.get('notes', '')
            if status in ('EXACT', 'APPROXIMATED'):
                require(all(rule.get(k) for k in ('source', 'logic', 'evidence', 'observation_logic')),
                        name + ': source, calculation evidence and observation logic are required.')
                require(callable(rule.get('builder')), name + ': executable historical calculation is missing.')
                if status == 'APPROXIMATED':
                    require(bool(rule.get('approximation')), name + ': describe the approximation explicitly.')
        rows.append(row)
    audit = pd.DataFrame(rows)
    counts = audit.HISTORICAL_RECONSTRUCTION_STATUS.value_counts().reindex(
        ['EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'], fill_value=0)
    require(len(audit) == int(counts.sum()) == 49, 'Reconstruction counts must sum to 49.')
    return audit, counts, extra


def require_reconstruction(audit, rules):
    blocked = audit.loc[~audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']), 'FEATURE_NAME'].tolist()
    require(not blocked, 'Historical reconstruction blocked. Supply verified V63 calculations and coverage for: ' + ', '.join(blocked))
    require(set(rules) == set(audit.FEATURE_NAME), 'Exactly 49 historical rules are required.')


def construct_sequence(grid, features, audit, rules, representation, encoding_contract):
    require_reconstruction(audit, rules)
    keys = ['PATIENT_ID', 'END_DT', 'TIME_STEP']
    # Builders receive dates and identifiers only, never RESP or split membership.
    request = grid.drop(columns='RESP').copy()
    values, observed = [], []
    provenance = []
    for feature in features:
        rule = rules[feature]
        result = rule['builder'](request.copy(), representation)
        needed = keys + ['VALUE', 'IS_OBSERVED', 'MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE',
                         'OBSERVATION_EVIDENCE', 'PROVENANCE_KIND', 'PROVENANCE_NOTE']
        require(isinstance(result, pd.DataFrame) and set(needed).issubset(result.columns), feature + ': incomplete historical output.')
        result = result[needed].copy()
        require(not result[keys].isna().any().any() and not result.duplicated(keys).any(), feature + ': invalid historical keys.')
        require(len(result) == len(grid), feature + ': historical output must cover every requested position explicitly.')
        aligned = request.merge(result, on=keys, how='left', validate='one_to_one', indicator=True)
        require(aligned._merge.eq('both').all(), feature + ': missing historical keys.')
        require(aligned.IS_OBSERVED.isin([0, 1, False, True]).all(), feature + ': observation status is required for every position.')
        known = aligned.IS_OBSERVED.astype(bool).to_numpy()
        require(aligned.OBSERVATION_EVIDENCE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(),
                feature + ': availability must be evidenced, including unavailable periods.')
        for field in ('MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE'):
            dates = pd.to_datetime(aligned[field], errors='raise')
            require(dates.dt.tz is None and dates.dropna().eq(dates.dropna().dt.normalize()).all(),
                    feature + ': provenance dates must be exact dates; timestamp rules require explicit review.')
            require((dates.isna() | dates.le(pd.to_datetime(aligned.PERIOD_END))).all(), feature + ': future information detected in ' + field)
        numbers = pd.to_numeric(aligned.VALUE, errors='raise').to_numpy(dtype=np.float64)
        require(np.isfinite(numbers[known]).all(), feature + ': observed values must be finite; explicitly calculate observed zeros.')
        require(np.isnan(numbers[~known]).all(), feature + ': unavailable feature history must remain null before padding.')
        kinds = aligned.PROVENANCE_KIND
        require(kinds.isin(['EVENT_DERIVED', 'OBSERVED_EMPTY', 'STATIC', 'UNAVAILABLE']).all(), feature + ': invalid provenance kind.')
        require(aligned.PROVENANCE_NOTE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(), feature + ': missing provenance explanation.')
        require(np.array_equal(kinds.ne('UNAVAILABLE').to_numpy(), known), feature + ': provenance and availability disagree.')
        event_rows = kinds.eq('EVENT_DERIVED')
        require(aligned.loc[event_rows, ['MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE']].notna().all().all(),
                feature + ': event-derived values need both event and availability date maxima.')
        require(np.all(numbers[kinds.eq('OBSERVED_EMPTY')] == 0), feature + ': observed-empty provenance requires a defined zero value.')
        if kinds.eq('STATIC').any():
            require(rule.get('static_feature') is True and bool(rule.get('static_rationale')),
                    feature + ': static provenance requires a verified static-feature rule and rationale.')
        values.append(numbers)
        observed.append(known)
        provenance.append({'feature': feature, 'rule': {k: v for k, v in rule.items() if k != 'builder'},
                           'builder_sha256': hashlib.sha256(marshal.dumps(rule['builder'].__code__)).hexdigest(),
                           'observed_positions': int(known.sum())})
    raw = np.column_stack(values)
    known = np.column_stack(observed)
    # A token is fully observed only when all fixed 49 inputs are supported at this cutoff.
    # Partial feature availability is reported, not disguised as no activity.
    valid = known.all(axis=1)
    long = grid.copy()
    long[features] = raw
    long['AVAILABLE_FEATURE_COUNT'] = known.sum(axis=1)
    long['IS_VALID_TIMESTEP'] = valid.astype('int64')
    long['IS_PADDED'] = (~valid).astype('int64')
    long['PADDING_REASON'] = np.where(valid, '', 'Insufficient evidenced history for one or more fixed features')
    # Preserve raw missingness for review; only the model tensor receives padding zeros.
    n_steps = 12 if representation == 'MONTHLY' else 4
    encoded = transform_fixed_v63(raw, features, encoding_contract['parameters'], encoding_contract['parameter_sha256'])
    require(np.array_equal(np.isnan(encoded), ~known), 'Business encoding changed historical availability.')
    X = np.where(valid[:, None], encoded, 0).astype(np.float32).reshape(-1, n_steps, 49)
    mask = valid.reshape(-1, n_steps)
    require(np.isfinite(X).all(), 'NaN/Inf after padding.')
    require(np.array_equal(mask, long.IS_VALID_TIMESTEP.to_numpy().reshape(mask.shape)), 'Mask alignment failure.')
    observed_zero = known.all(axis=1) & np.all(raw == 0, axis=1)
    require(valid[observed_zero].all(), 'Observed zero activity was incorrectly masked.')
    return {'X': X, 'valid': mask, 'long': long, 'known': known, 'encoded': encoded, 'provenance': provenance,
            'representation': representation}


def sparsity_report(bundle, features):
    raw = bundle['long'][features].to_numpy(dtype=float)
    known = bundle['known']
    valid = bundle['valid']
    counts = known.sum(axis=0)
    zeros = ((raw == 0) & known).sum(axis=0)
    feature = pd.DataFrame({'FEATURE_NAME': features, 'OBSERVED_VALUES': counts,
                            'UNAVAILABLE_VALUES': (~known).sum(axis=0), 'OBSERVED_ZERO_VALUES': zeros,
                            'OBSERVED_ZERO_PERCENT': np.divide(100. * zeros, counts, out=np.full(49, np.nan), where=counts > 0)})
    encoded_zeros = ((bundle['encoded'] == 0) & known).sum(axis=0)
    feature['V63_ENCODED_ZERO_PERCENT_OBSERVED'] = np.divide(
        100. * encoded_zeros, counts, out=np.full(49, np.nan), where=counts > 0)
    eligible = int(known.sum())
    report = {'MODEL': bundle['representation'], 'TOTAL_TIMESTEPS': int(valid.size),
              'VALID_TIMESTEPS': int(valid.sum()), 'PADDED_TIMESTEPS': int((~valid).sum()),
              'VALID_PERCENT': float(valid.mean() * 100), 'PADDED_PERCENT': float((~valid).mean() * 100),
              'ALL_PADDED_SNAPSHOTS': int((~valid.any(axis=1)).sum()),
              'OBSERVED_FEATURE_ZERO_PERCENT': float(((raw == 0) & known).sum() * 100 / eligible) if eligible else None,
              'V63_ENCODED_FEATURE_ZERO_PERCENT_OBSERVED': float(encoded_zeros.sum() * 100 / eligible) if eligible else None,
              'TENSOR_ZERO_PERCENT_INCLUDING_PADDING': float((bundle['X'] == 0).mean() * 100),
              'OBSERVED_ZERO_TIMESTEPS': int((valid.reshape(-1) & np.all(raw == 0, axis=1)).sum())}
    distribution = pd.Series(valid.sum(axis=1)).value_counts().sort_index().rename_axis('VALID_TIMESTEPS').reset_index(name='SNAPSHOTS')
    return report, feature, distribution


def display_sequence(bundle, features):
    frame = bundle['long'].copy()
    if bundle['representation'] == 'MONTHLY':
        frame = frame.rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'})
    else:
        frame = frame.rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'})
    print(bundle['representation'], 'raw historical values; unavailable values are null here and zero-padded only in the model tensor')
    for label in (0, 1):
        sample = frame.loc[frame.RESP.eq(label)].head(24)
        if not sample.empty:
            print('RESP =', label)
            display(sample)
    for state in (1, 0):
        sample = frame.loc[frame.IS_VALID_TIMESTEP.eq(state)].head(12)
        print('Valid' if state else 'Padded', 'positions:', 'available' if not sample.empty else 'none in this dataset')
        if not sample.empty:
            display(sample)
    raw = bundle['long']
    summary = raw.groupby(['PATIENT_ID', 'END_DT'], sort=True).agg(
        VALID=('IS_VALID_TIMESTEP', 'sum'), PADDED=('IS_PADDED', 'sum')).reset_index()
    activity = raw[features].fillna(0).ne(0).sum(axis=1)
    summary['NONZERO_VALUES'] = activity.groupby([raw.PATIENT_ID, raw.END_DT]).sum().to_numpy()
    choices = [('relatively dense', summary.sort_values(['VALID', 'NONZERO_VALUES'], ascending=False).head(1)),
               ('partially sparse', summary.loc[summary.VALID.gt(0)].sort_values('NONZERO_VALUES').head(1)),
               ('substantial padding', summary.loc[summary.PADDED.gt(0)].sort_values('PADDED', ascending=False).head(1))]
    for title, example in choices:
        print('Structural example:', title, '(selected without model scores)')
        if example.empty:
            print('No matching example in this dataset.')
        else:
            r = example.iloc[0]
            display(frame.loc[frame.PATIENT_ID.eq(r.PATIENT_ID) & frame.END_DT.eq(r.END_DT)])


def prepared_blobs(metadata, features, bundles, audit, comparison, snapshot_X, encoding_contract):
    report = {'schema': 3, 'dataset_id': DATASET_ID, 'features': features,
              'business_encoding': encoding_contract,
              'population_sha256': digest_json(snapshot_records(metadata)),
              'snapshot_feature_sha256': array_hash(snapshot_X),
              'configuration_audit': comparison, 'audit': audit.astype(object).where(pd.notna(audit), None).to_dict('records'),
              'orientation': '0=newest; calendar buckets; most recent bucket truncated at END_DT',
              'implementation_sha256': IMPLEMENTATION_SHA256,
              'representations': {}}
    artifacts = {'population.json': canonical_json(snapshot_records(metadata)).encode()}
    for name, b in bundles.items():
        buffer = io.BytesIO()
        np.savez_compressed(buffer, X=b['X'], valid=b['valid'])
        artifacts[name + '.npz'] = buffer.getvalue()
        sparsity, _, distribution = sparsity_report(b, features)
        report['representations'][name] = {'shape': list(b['X'].shape), 'X_sha256': array_hash(b['X']),
              'valid_sha256': array_hash(b['valid']), 'sparsity': sparsity,
              'valid_distribution': distribution.to_dict('records'), 'provenance': b['provenance']}
    artifacts['manifest.json'] = canonical_json(report).encode()
    return artifacts


PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}











def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value

def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'

def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out

def align_features(snapshots, model_data, features):
    features = parse_features(features)
    expected = normalize_metadata(snapshots).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    actual = normalize_metadata(model_data)
    missing = set(features).difference(model_data.columns)
    if missing:
        raise ValueError("Selected columns absent from MODEL_DATA: " + repr(sorted(missing)))
    source = actual.copy()
    for name in features:
        # Decimal fractions are converted to float, never through an integer cast.
        source[name] = pd.to_numeric(model_data[name], errors="raise").to_numpy(dtype=np.float64)
    aligned = expected.merge(source, on=["PATIENT_ID", "END_DT"], how="left",
                             validate="one_to_one", suffixes=("", "_SOURCE"), indicator=True)
    if not aligned._merge.eq("both").all() or not aligned.RESP.eq(aligned.RESP_SOURCE).all():
        raise ValueError("Missing source keys or conflicting labels in MODEL_DATA.")
    X = aligned[features].to_numpy(dtype=np.float64)
    if np.isinf(X).any():
        raise ValueError("Infinite source feature values.")
    return expected, X



def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())

def read_query(query):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load())

def fetch_source_features(features):
    fields = ["PATIENT_ID", "END_DT", "RESP"] + features
    selected = ", ".join("M." + quote_identifier(f) for f in fields)
    # Select only the frozen cohort. Duplicated source keys remain visible and fail validation.
    query = (f"SELECT {selected} FROM {DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA M "
             f"INNER JOIN (SELECT DISTINCT PATIENT_ID, END_DT FROM {DATABASE}.DS_ML.{PREFIX}_SNAPSHOTS) S "
             "ON M.PATIENT_ID = S.PATIENT_ID AND M.END_DT = S.END_DT")
    return read_query(query).toPandas()

def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")
import base64
def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]

"""Pure fixed-parameter transformations and date boundaries; no warehouse IO.

These helpers do not reconstruct a feature, choose predictors, infer source
coverage, fit percentiles, infer feature types, or select a modeling population.
"""

import math
from collections.abc import Sequence



def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(len(result) == 49, 'Exactly 49 authoritative predictors are required.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == 49,
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (49 ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    49-feature input list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)


def transform_fixed_v63(raw, features, parameters, expected_parameter_hash=None):
    """Transform any numeric array whose last axis is the fixed ordered 49.

    AGE: ceil(raw / 10).
    Binary: 1 exactly when raw > 0, else 0.
    Numeric: 1 when raw > VALUE_P, else raw / (VALUE_P + 1).
    Null raw values stay NaN. No filling, scaling fit, clipping below zero,
    percentile calculation, ranking or row/feature selection occurs here.
    """
    features = _v63_features(features)
    _v63_require(isinstance(parameters, (list, tuple)) and len(parameters) == 49,
                 'Exactly 49 frozen parameter records are required.')
    _v63_require(all(isinstance(row, dict) for row in parameters), 'Invalid parameter record.')
    required_keys = {'FEATURE_ORDER', 'FEATURES', 'VALUE_P', 'VAR_TYP', 'TRANSFORM'}
    _v63_require(all(set(row) == required_keys for row in parameters),
                 'Frozen parameter record fields changed.')
    _v63_require([row['FEATURES'] for row in parameters] == features,
                 'Parameter feature order differs from the authoritative list.')
    _v63_require([row['FEATURE_ORDER'] for row in parameters] == list(range(49)),
                 'Parameter FEATURE_ORDER must be exactly 0 through 48.')
    # Revalidate fixed values and transformation precedence; do not trust a
    # modified or hand-assembled parameter manifest merely because it has 49 rows.
    canonical, digest = fixed_v63_parameters(features, pd.DataFrame(parameters))
    _v63_require(list(parameters) == canonical, 'Frozen parameter values or transform rules changed.')
    if expected_parameter_hash is not None:
        _v63_require(isinstance(expected_parameter_hash, str) and digest == expected_parameter_hash,
                     'Frozen FEATURES_SUMMARY parameter fingerprint changed.')
    array = np.asarray(raw)
    _v63_require(array.ndim >= 1 and array.shape[-1] == 49,
                 'Input last axis must contain the 49 authoritative features.')
    _v63_require(not np.iscomplexobj(array), 'Complex raw feature values are invalid.')
    if array.dtype == object:
        _v63_require(not any(isinstance(value, (complex, np.complexfloating)) for value in array.flat),
                     'Complex raw feature values are invalid.')
        # Preserve nullable pandas/scalar missing values before numeric conversion.
        array = np.where(pd.isna(array), np.nan, array)
    try:
        values = np.array(array, dtype=np.float64, copy=True)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Raw features must contain real numbers or nulls.') from error
    _v63_require(not np.isinf(values).any(), 'Infinite raw feature values are invalid.')
    out = np.full(values.shape, np.nan, dtype=np.float64)
    for index, row in enumerate(canonical):
        column = values[..., index]
        available = ~np.isnan(column)
        if row['TRANSFORM'] == 'AGE_CEIL_DECADE':
            transformed = np.ceil(column / 10.0)
        elif row['TRANSFORM'] == 'BINARY_POSITIVE':
            transformed = (column > 0).astype(np.float64)
        else:
            cap = row['VALUE_P']
            transformed = np.where(column > cap, 1.0, column / (cap + 1.0))
        out[..., index] = np.where(available, transformed, np.nan)
    _v63_require(not np.isinf(out).any(), 'Fixed transformation produced infinite values.')
    _v63_require(np.array_equal(np.isnan(out), np.isnan(values)),
                 'Fixed transformation must preserve the raw null mask.')
    return out


def _v63_date(value, label='cutoff'):
    try:
        date = pd.Timestamp(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(label + ' must be an exact date.') from error
    _v63_require(not pd.isna(date), label + ' must be explicitly supplied.')
    _v63_require(date.tzinfo is None and date == date.normalize(),
                 label + ' must be a timezone-naive date, not an intraday timestamp.')
    return date


def v63_rolling_year_start(end):
    """DATEADD(year, -1, end) + 1 day, including leap-day clamping."""
    cutoff = _v63_date(end)
    return cutoff - pd.DateOffset(years=1) + pd.Timedelta(days=1)


def v63_recent_inclusive_window(end):
    """Inclusive DATEADD(month,-4,end) through DATEADD(month,-1,end).

    These are shifted dates, not first/last boundaries of calendar months.
    """
    cutoff = _v63_date(end)
    return cutoff - pd.DateOffset(months=4), cutoff - pd.DateOffset(months=1)


def v63_custom_l3m_inclusive_window(end):
    """Inclusive end-90 days through end (91 possible calendar dates)."""
    cutoff = _v63_date(end)
    return cutoff - pd.Timedelta(days=90), cutoff


def v63_hcp_730_day_start(end):
    """Return end-730 days; source SQL must retain its own boundary operators."""
    return _v63_date(end) - pd.Timedelta(days=730)


def v63_adherence_270_day_start(*, effective_data_end):
    """Return the supplied effective-data-end anchor minus 270 days.

    A snapshot cutoff is not an implicit substitute for effective_data_end.
    This helper creates a date boundary only; it does not calculate adherence.
    """
    return _v63_date(effective_data_end, 'effective_data_end') - pd.Timedelta(days=270)


"""User-supplied V63 lineage and frozen business encodings; no feature selection."""
SOURCE_REGISTRY = [
    ('MEDICAL', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST', 'SERVICE_DATE', 'DX/PX claims; current extract, no historical availability timestamp supplied'),
    ('PHARMACY', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST', 'FILL_DATE', 'RX claims; transaction and counting rules still require column-level mapping'),
    ('PROVIDERS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PROVIDERS_LATEST', '', 'Current specialty/type; not a historical as-of dimension'),
    ('PLANS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PLANS_LATEST', '', 'Current insurance group/segment; not a historical as-of dimension'),
    ('DEMOGRAPHICS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PATIENT_DEMOGRAPHICS_LATEST', '', 'Year of birth for year-based age; patient/YOB columns must be verified'),
    ('CODE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID', '', 'Code descriptions, hierarchy and chronic flags'),
    ('ANNUAL_CODE_COUNTS', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID_ANNUAL_CNT', '', 'Upstream inclusion lineage only; do not rerun selection'),
    ('PROCEDURE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.ALL_PROCEDURES_SIMPLE', '', 'ICD-10-PCS decomposition; not a substitute for visit-sequence features'),
    ('HCP_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.GATREX_TRIGGER.HCP_LOOKUP', '', 'Active NPI/specialty; historical version not supplied'),
    ('DRUG_BASKET', 'TAK861.NARCOLEPSY_MARKET_BASKET_CODES', '', 'Database not supplied; code columns and inclusion rules still required'),
    ('UPSTREAM_UNIVERSE', 'TAK861.TAK861_TX_READY_PATIENT_UNIVERSE_V8_CF', '', 'Lineage only; never substitute for frozen V63 snapshots'),
    ('UPSTREAM_BACKTEST', 'TAK861.TAK861_TX_READY_BACKTEST_UNIVERSE_V8_CF', '', 'Lineage only; never replace the existing TEST assignment'),
]


def inspect_source_schemas():
    rows = []
    for kind, table, date_column, note in SOURCE_REGISTRY:
        if table.count('.') != 2:
            rows.append({'SOURCE': kind, 'TABLE': table, 'COLUMN': None, 'DATA_TYPE': None,
                         'STATUS': 'DATABASE_UNRESOLVED', 'NOTES': note})
            continue
        try:
            schema = read_table(table).schema
            rows.extend({'SOURCE': kind, 'TABLE': table, 'COLUMN': field.name,
                         'DATA_TYPE': field.dataType.simpleString(), 'STATUS': 'SCHEMA_READ', 'NOTES': note}
                        for field in schema.fields)
        except Exception as error:
            # A metadata-discovery failure is displayed; it cannot enable reconstruction.
            # Do not print a connector exception that might contain connection details.
            rows.append({'SOURCE': kind, 'TABLE': table, 'COLUMN': None, 'DATA_TYPE': None,
                         'STATUS': type(error).__name__, 'NOTES': note + '; metadata unavailable on this runtime'})
    return pd.DataFrame(rows)


def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'max_f_50': 'Ceiling only; RND explanation remains a hypothesis until ranked rows establish it'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    required = set(['PATIENT_ID', 'END_DT', 'RESP'] + features)
    require(required.issubset(read_table(SOURCE_PREFIX + '_MODEL_DATA').columns), 'MODEL_DATA lacks configured columns.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'raw reconstructed features', 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract


def display_feature_sources(features, summary, parameters):
    allowed = ['FEATURES', 'SEQ', 'F_FLG', 'VAR_TYP', 'VALUE_P', 'VALUE_CNT', 'RVALUE', 'RR_RATIO', 'IMPORTANCE', 'IMPORTANCE_MOD']
    selected = pd.DataFrame({'FEATURES': features, 'FEATURE_ORDER': range(49)}).merge(
        summary[[c for c in allowed if c in summary.columns]], on='FEATURES', how='left', validate='one_to_one')
    display(selected)
    if {'SEQ', 'F_FLG'}.issubset(summary.columns):
        seq = pd.to_numeric(summary.SEQ, errors='raise')
        ranked = summary.loc[seq.le(50), [c for c in allowed if c in summary.columns]].copy()
        ranked['SEQ'] = seq.loc[ranked.index]
        ranked['IN_FIXED_49'] = ranked.FEATURES.isin(features)
        print('Original top-50 metadata for reconciliation only; no feature selection is performed:')
        display(ranked.sort_values('SEQ'))
        dropped = ranked.loc[ranked.F_FLG.astype(str).str.lower().eq('dropped')]
        print('Dropped rows within the original ceiling (do not assume RND is the dictionary discrepancy):')
        display(dropped)
    print('Counts/ratios can legitimately have Binary encoding: VAR_TYP is the learned V63 output type.')
    print('Frozen VALUE_P/VAR_TYP are reused, never fitted with current validation/test labels; their upstream fit population is unverified.')


def documented_feature_windows(request):
    out = request[['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END']].copy()
    # This is a date-boundary display, not evidence of patient observation coverage.
    boundaries = []
    for value in request.PERIOD_END:
        recent_start, recent_end = v63_recent_inclusive_window(value)
        l3m_start, l3m_end = v63_custom_l3m_inclusive_window(value)
        boundaries.append({
            'PROCEDURE_L12M_START_INCLUSIVE': v63_rolling_year_start(value).strftime('%Y-%m-%d'),
            'PROCEDURE_L12M_END_INCLUSIVE': value,
            'GENERIC_RECENT_START_INCLUSIVE': recent_start.strftime('%Y-%m-%d'),
            'GENERIC_RECENT_END_INCLUSIVE': recent_end.strftime('%Y-%m-%d'),
            'CUSTOM_L3M_START_INCLUSIVE': l3m_start.strftime('%Y-%m-%d'),
            'CUSTOM_L3M_END_INCLUSIVE': l3m_end.strftime('%Y-%m-%d'),
            'HCP_AT_730_DAY_START_INCLUSIVE': v63_hcp_730_day_start(value).strftime('%Y-%m-%d'),
            'ADHERENCE_ANCHOR': 'effective_data_end not supplied; do not substitute END_DT',
        })
    for key in boundaries[0] if boundaries else []:
        out[key] = [row[key] for row in boundaries]
    return out


def feature_lineage_notes(name):
    if name == 'AGE':
        return ('DEMOGRAPHICS: patient year of birth', 'Raw age = cutoff year minus birth year; encoded AGE = CEIL(raw age/10). YOB source column/availability must be verified.')
    if name.startswith('MAX_AT_') or name.startswith('AVG_AT_'):
        return ('PHARMACY + PROVIDERS/HCP_REFERENCE', 'Provider AT usage lookback: cutoff minus 730 days through cutoff. Patient-to-HCP linkage, denominator, NTILE population/averaging and drug exclusions remain unresolved.')
    if name.startswith('_'):
        return ('Custom visit-sequence logic not established', 'Dictionary visit sequences are not ICD-10-PCS character decomposition; visit grain, order and within-five-visits rule are still required.')
    if name in {'TIMES_GENERIC_MIX_ADJUSTED', 'UNIQUE_GENERICS_TRIED', 'NUM_DISCONTINUATIONS', 'NUM_GAPS_30_PLUS_DAYS'}:
        return ('PHARMACY + DRUG_BASKET; custom feature branch', 'Custom lifetime/adherence/treatment logic coexists with procedure features; 270-day adherence anchor is effective_data_end, not automatically END_DT. Exact event/gap/overlap/generic rules remain unresolved.')
    if name.endswith('_L3M'):
        return ('MEDICAL + CODE_REFERENCE; custom feature branch', 'Custom L3M: cutoff minus 90 days through cutoff, inclusive. Diagnosis sets and counting/deduplication grain must be verified.')
    if name.endswith('_L12M') or name == 'L12M_NARCO_CLAIMS':
        return ('MEDICAL + CODE_REFERENCE; rolling feature', 'L12M: DATEADD(year,-1,cutoff)+1 day through cutoff, inclusive. Qualifying codes and count grain still required.')
    if name == 'L6M_NARCO_CLAIMS':
        return ('MEDICAL + CODE_REFERENCE; custom feature branch', 'Six-month presence is described; exact calendar/day boundary, code set and coverage remain unresolved.')
    if name in {'NARCO_CLAIMS_RECENT_RATIO', 'SLEEP_MED_VISIT_RECENCY_PCT'}:
        return ('Custom numerator/denominator source must be verified', 'Do not substitute generic RECENT_PCT windows for a custom ratio; numerator/denominator and zero-denominator rules remain unresolved.')
    if 'RECENT_PCT' in name:
        return ('Generic recency branch or custom branch; resolve lineage', 'Generic RECENT_PCT uses DATEADD(month,-4,cutoff) through DATEADD(month,-1,cutoff); earlier description also requires value>6 and recent>2. Verify actual branch/denominator before implementing.')
    source = 'MEDICAL/CODE_REFERENCE' if name.startswith(('CPT_', 'DX_', 'PL_', 'VISIT_', 'LEVEL_')) else 'PHARMACY/CODE_REFERENCE'
    if 'SPECIALIST' in name or name.startswith('HCPS_'):
        source += ' + PROVIDERS/HCP_REFERENCE (current state)'
    if any(word in name for word in ('INSURANCE', 'PAYER')):
        source += ' + PLANS (current state)'
    return (source, 'Procedure branch uses inclusive one-year window at each cutoff; confirm branch, code mapping, qualifying statuses and count/flag grain. Current metadata is not historically versioned.')


def _age_require(condition, message):
    if not condition:
        raise ValueError(message)


def _validated_age_demographics(frame):
    _age_require(isinstance(frame, pd.DataFrame), 'AGE source must be a DataFrame.')
    _age_require({'PATIENT_ID', 'YEAR_OF_BIRTH'}.issubset(frame.columns),
                 'AGE source requires PATIENT_ID and YEAR_OF_BIRTH.')
    _age_require(not frame.columns.duplicated().any(), 'Duplicate AGE source columns.')
    result = frame[['PATIENT_ID', 'YEAR_OF_BIRTH']].copy()
    _age_require(result.PATIENT_ID.map(lambda value: isinstance(value, str) and bool(value.strip())).all(),
                 'AGE source patient identifiers must be nonempty strings.')
    source_years = result.YEAR_OF_BIRTH
    source_missing = source_years.isna().to_numpy()
    _age_require(not source_years.map(lambda value: isinstance(value, (bool, np.bool_, complex, np.complexfloating))).any(),
                 'YEAR_OF_BIRTH must be a real year, not boolean or complex.')
    try:
        numeric = pd.to_numeric(source_years, errors='raise').astype('float64').to_numpy()
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Invalid YEAR_OF_BIRTH values.') from error
    _age_require(np.isfinite(numeric[~source_missing]).all(),
                 'Nonmissing YEAR_OF_BIRTH must be finite.')
    _age_require(np.isnan(numeric[source_missing]).all(), 'Missing YEAR_OF_BIRTH conversion changed.')
    known = numeric[~source_missing]
    _age_require(np.equal(known, np.floor(known)).all() and np.greater_equal(known, 0).all(),
                 'YEAR_OF_BIRTH must be a nonnegative integer year.')
    result['YEAR_OF_BIRTH'] = numeric
    conflicts = result.groupby('PATIENT_ID', sort=False).YEAR_OF_BIRTH.nunique(dropna=False).gt(1)
    _age_require(not conflicts.any(),
                 'Conflicting demographic YEAR_OF_BIRTH rows for a patient; no arbitrary row is selected.')
    # Only identical patient/year rows may collapse. Mixed known/null duplicates
    # are rejected above rather than silently preferring a known year.
    return result.drop_duplicates('PATIENT_ID').reset_index(drop=True)


def register_age_rules(authoritative_features, read_query_fn, database, prefix):
    """Return an AGE rule only when AGE is an authoritative predictor.

    Invoke this factory in the notebook's registration cell on every run. Its
    cache is private to that invocation, source, reader and frozen-cohort query;
    no cache survives a new registration or is shared across notebook sessions.
    """
    _age_require(callable(read_query_fn), 'AGE source reader must be callable.')
    _age_require(isinstance(authoritative_features, (list, tuple, np.ndarray)),
                 'Authoritative feature order must be an explicit sequence.')
    features = list(authoritative_features)
    _age_require(all(isinstance(name, str) and name for name in features), 'Invalid authoritative feature names.')
    _age_require(len(features) == len(set(features)), 'Duplicate authoritative features.')
    if 'AGE' not in features:
        return {}
    _age_require(all(isinstance(value, str) and re.fullmatch(r'[A-Z][A-Z0-9_]*', value)
                     for value in (database, prefix)), 'Invalid AGE source/cohort identifiers.')
    demographics_table = 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PATIENT_DEMOGRAPHICS_LATEST'
    population_table = database + '.DS_ML.' + prefix + '_SNAPSHOTS'
    query = (
        'SELECT s.PATIENT_ID, d.YEAR_OF_BIRTH '
        'FROM (SELECT DISTINCT PATIENT_ID FROM ' + population_table + ') s '
        'LEFT JOIN ' + demographics_table + ' d ON s.PATIENT_ID = d.PATIENT_ID'
    )
    cached_demographics = None

    def age_builder(request, representation):
        nonlocal cached_demographics
        _age_require(representation in ('MONTHLY', 'QUARTERLY'), 'Unexpected AGE representation.')
        _age_require(isinstance(request, pd.DataFrame), 'AGE request must be a DataFrame.')
        keys = ['PATIENT_ID', 'END_DT', 'TIME_STEP']
        _age_require(set(keys + ['PERIOD_END']).issubset(request.columns), 'AGE request lacks required keys/cutoff.')
        _age_require(not request[keys + ['PERIOD_END']].isna().any().any()
                     and not request.duplicated(keys).any(), 'Invalid or duplicate AGE request keys.')
        _age_require(request.PATIENT_ID.map(lambda value: isinstance(value, str) and bool(value.strip())).all(),
                     'AGE request identifiers must be nonempty strings.')
        cutoffs = pd.to_datetime(request.PERIOD_END, errors='raise')
        snapshots = pd.to_datetime(request.END_DT, errors='raise')
        _age_require(cutoffs.dt.tz is None and snapshots.dt.tz is None
                     and cutoffs.eq(cutoffs.dt.normalize()).all()
                     and snapshots.eq(snapshots.dt.normalize()).all(), 'AGE cutoffs must be exact dates.')
        _age_require(cutoffs.le(snapshots).all(), 'AGE timestep cutoff exceeds snapshot cutoff.')
        if cached_demographics is None:
            loaded = read_query_fn(query)
            frame = loaded if isinstance(loaded, pd.DataFrame) else loaded.toPandas()
            cached_demographics = _validated_age_demographics(frame)
        # Index lookup preserves request order and leaves missing patients/YOB null.
        lookup = cached_demographics.set_index('PATIENT_ID').YEAR_OF_BIRTH
        years = request.PATIENT_ID.map(lookup).to_numpy(dtype=np.float64)
        observed = np.isfinite(years)
        cutoff_years = cutoffs.dt.year.to_numpy(dtype=np.float64)
        _age_require(np.less_equal(years[observed], cutoff_years[observed]).all(),
                     'YEAR_OF_BIRTH is after a requested historical cutoff; cannot create a negative age.')
        values = np.where(observed, cutoff_years - years, np.nan)
        result = request[keys].copy()
        result['VALUE'] = values
        result['IS_OBSERVED'] = observed.astype(np.int64)
        # Birth year is an assumed stable attribute, not an event at PERIOD_END.
        # Do not fabricate a historical source-availability timestamp.
        result['MAX_EVENT_DATE'] = pd.NaT
        result['MAX_AVAILABLE_DATE'] = pd.NaT
        result['PROVENANCE_KIND'] = np.where(observed, 'STATIC', 'UNAVAILABLE')
        result['PROVENANCE_NOTE'] = np.where(
            observed,
            'Raw age is derived from stable birth year; current demographic extract has no verified historical availability date.',
            'YEAR_OF_BIRTH is missing; historical age is unavailable and is not filled with zero.')
        result['OBSERVATION_EVIDENCE'] = np.where(
            observed,
            'Current demographic YEAR_OF_BIRTH is present and treated as stable; historical availability is an explicit approximation.',
            'No usable YEAR_OF_BIRTH in the cohort-restricted current demographic source.')
        return result

    return {'AGE': {
        'status': 'APPROXIMATED',
        'source': demographics_table + '.YEAR_OF_BIRTH; cohort restricted by ' + population_table,
        'logic': 'Raw AGE = year(PERIOD_END) - YEAR_OF_BIRTH; no bucketing or cap transformation in this builder.',
        'evidence': 'Supplied demographic source/join and year-based age formula.',
        'observation_logic': 'Known stable YEAR_OF_BIRTH permits an age value; missing birth year is unavailable. This does not establish clinical observation coverage.',
        'approximation': 'YEAR_OF_BIRTH is taken from the current demographic extract; its historical availability/corrections are not verified.',
        'static_feature': True,
        'static_rationale': 'The source birth year is treated as stable; derived age varies with each timestep cutoff year. No event or availability date is invented.',
        'notes': 'The shared downstream V63 transform alone applies CEIL(raw age/10). The factory resets the private source cache on each registration.',
        'builder': age_builder,
    }}


def supplied_feature_reconciliation(features, supplied):
    require(len(supplied) == 49, 'The supplied review must contain 49 distinct candidate names.')
    require(len(features) == len(set(features)) == 49, 'Exactly 49 authoritative predictors required.')
    rows = []
    for order, name in enumerate(features):
        rows.append({'FEATURE_NAME': name, 'FEATURE_ORDER': order,
                     'IN_AUTHORITATIVE_V63': True, 'IN_SUPPLIED_CODE': name in supplied,
                     'NAME_MATCH': 'MATCHED' if name in supplied else 'AUTHORITATIVE_ONLY'})
    for name in supplied:
        if name not in features:
            rows.append({'FEATURE_NAME': name, 'FEATURE_ORDER': None,
                         'IN_AUTHORITATIVE_V63': False, 'IN_SUPPLIED_CODE': True,
                         'NAME_MATCH': 'SUPPLIED_ONLY'})
    return pd.DataFrame(rows)


In [ ]:
# Read authoritative features and original snapshot population
features, configuration_audit = ordered_features()
business_parameters, feature_summary, encoding_contract = load_business_parameters(features)
display_feature_sources(features, feature_summary, business_parameters)
display(pd.DataFrame(business_parameters))
snapshots = read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()
metadata, snapshot_X = align_features(snapshots, fetch_source_features(features), features)
population_check(metadata)
print('Confirmed authoritative feature count:', len(features))
print('Snapshot feature matrix:', snapshot_X.shape)
display(pd.DataFrame([configuration_audit]))
for label in (0, 1):
    print('Original population examples: RESP =', label)
    display(metadata.loc[metadata.RESP.eq(label)].head(5))
display(pd.DataFrame({'FEATURE_ORDER': range(49), 'FEATURE_NAME': features,
    'SNAPSHOT_MISSING_PERCENT': np.isnan(snapshot_X).mean(axis=0) * 100,
    'SNAPSHOT_ZERO_PERCENT_ALL_ROWS': (snapshot_X == 0).mean(axis=0) * 100}))
print('Snapshot missingness and feature zeros are distinct from unavailable historical timesteps.')


### Historical calculation and coverage requirements
`HISTORICAL_RULES` still requires executable source calculations: the supplied encoding and date windows alone cannot establish raw claim counts, custom metrics or patient coverage. Add verified V63 raw calculation functions **inside this notebook**, then register each authoritative feature with:

- `status`: EXACT or APPROXIMATED; `source`, `logic`, `evidence`, `observation_logic`, `notes`, and `builder`; APPROXIMATED also requires `approximation`.
- `builder(request, representation)` receives PATIENT_ID, original END_DT, TIME_STEP, PERIOD_START, PERIOD_END; it never receives RESP or split assignments. Return raw, unencoded VALUE exactly once per key, with IS_OBSERVED, MAX_EVENT_DATE, MAX_AVAILABLE_DATE, OBSERVATION_EVIDENCE, PROVENANCE_KIND, PROVENANCE_NOTE. The shared constructor applies the frozen V63 transformation after raw reconstruction; builders must not apply it themselves.
- Recompute each feature as of PERIOD_END, using its actual rolling/calendar/day window, numerator/denominator policy, visit order, lifecycle statuses, mappings and exclusions. For period-defined measures use PERIOD_START as defined by verified business logic. Quarterly builders receive their own three-month boundaries and must recalculate; they do not receive monthly feature values to sum.
- IS_OBSERVED must come from verified source coverage **over the entire lookback needed by that feature**, including gaps, source extract availability and late-arriving information. It cannot be inferred from first/last claim or nonzero values. Event and availability maxima must describe all source rows used; null maxima are permitted for a verified observed empty window or verified static calculation.
- PROVENANCE_KIND is EVENT_DERIVED (both date maxima required), OBSERVED_EMPTY (business-defined zero), STATIC (requires `static_feature=True` and `static_rationale` in the verified rule), or UNAVAILABLE (null value). PROVENANCE_NOTE explains the evidence; these declarations are not substitutes for reviewing the source calculation.
- An observed empty window may return zero only when its business definition says zero; undefined ratios and incomplete history must use an explicit approved rule. Unavailable feature history returns null VALUE and IS_OBSERVED=0 with an explanation. Missing calculation code is a global blocker, not patient-level padding.
- A timestep is valid only if all 49 values have evidenced historical availability; partial availability is shown with AVAILABLE_FEATURE_COUNT and masks the entire token. Raw displays retain nulls; model padding is zero. All-padded snapshots stay in the cohort and bypass attention to the unchanged head with a zero pooled vector.

Date helpers implement the supplied procedure/L12M year-minus-one-plus-one-day window, inclusive generic recency minus-four to minus-one months, inclusive custom L3M cutoff-minus-90 days, and inclusive 730-day HCP AT window. The separate 270-day adherence helper requires effective_data_end explicitly; no substitute anchor is assumed. A calendar bucket boundary is not a rolling-feature window or observation evidence. Quarterly features are recalculated at each quarterly cutoff with their original definitions, never a blind monthly sum.

These checks reject out-of-window provenance, but cannot prove a supplied formula is clinically correct: review the actual V63 logic, especially source availability and window boundaries. `_LATEST` metadata cannot satisfy historical availability merely by assigning the claim date to MAX_AVAILABLE_DATE. AGE now uses the supplied year-of-birth calculation at each cutoff. It is explicitly APPROXIMATED because the current demographic extract does not prove historical availability or unchanged birth-year records. Missing birth years remain unavailable; AGE does not establish clinical observation coverage. Other missing calculations or source availability continue to block reconstruction.


Supplied draft calculations are recorded as review metadata, not silently enabled. In particular, sequence visit-offset checks, adherence day/transition boundaries, the effective_data_end anchor, provider metric window availability and the narcolepsy-ratio denominator need resolution. Unknown observation coverage cannot be replaced with cohort membership or source readability. The review does not delete dictionary entries or change the configured feature set.


In [ ]:
# Dictionary transcription and explicit historical calculation registry
# Dictionary names/types support source reconciliation; V63 VAR_TYP controls encoding.
# Resolved window/encoding rules do not certify unknown raw formulas or historical coverage.
BUSINESS_DICTIONARY_REFERENCE = [
    {'seq': 1, 'name': 'MAX_AT_RX_NTILE_NOZOLP', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 2, 'name': 'CPT_95805_SLEEP_STUDY_MULTIPLE_TRIALS', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 3, 'name': 'AGE', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 4, 'name': 'TIMES_GENERIC_MIX_ADJUSTED', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 5, 'name': 'UNIQUE_GENERICS_TRIED', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 6, 'name': 'VA_CLASS1_CN809_CNS_STIMULANTS_OTHER', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 7, 'name': 'AVG_AT_PTS_NOZOLP', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 8, 'name': 'L12M_NARCO_CLAIMS', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 9, 'name': 'NUM_DISCONTINUATIONS', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 10, 'name': 'NARCO_CLAIMS_RECENT_RATIO', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 11, 'name': 'DX_G47411_NARCOLEPSY_WITH_CATAPLEXY', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 12, 'name': 'ATC_1_OTHER_ANTIDEPRESSANTS', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 13, 'name': 'NUM_SPECIALISTS', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 14, 'name': 'NUM_EXCESSIVE_DAYTIME_SLEEPINESS_CLAIMS_L12M', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 15, 'name': 'CPT_99204_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT', 'name_complete': False, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 16, 'name': 'MAX_AT_RX_NTILE_WITHZOLP', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 17, 'name': 'HCPS_RX_HCP_S1_PHYSICIAN_ASSISTANT', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 18, 'name': 'NUM_PSYCH_COMORBIDITIES_L12M', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 19, 'name': '_G47411_G47411_', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 20, 'name': 'SLEEP_MED_VISIT_RECENCY_PCT', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 21, 'name': 'NUM_GAPS_30_PLUS_DAYS', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'UNRESOLVED'},
    {'seq': 22, 'name': 'NUM_CATAPLEXY_CLAIMS_L3M', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 23, 'name': 'CLAIMS_RX_REJECTED', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 24, 'name': 'NUM_HYPERSOMNIA_CLAIMS_L3M', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 25, 'name': 'L6M_NARCO_CLAIMS', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 26, 'name': 'RX_INSURANCE_SEGMENT_INTEGRATED', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 27, 'name': 'RX_INSURANCE_SEGMENT_PBM', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 28, 'name': 'VA_CLASS1_CN801_AMPHETAMINES', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 29, 'name': 'PL_22', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 30, 'name': '_G4710_G47411_', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 31, 'name': 'LEVEL_2_CPT_OFFICE_OUTPATIENT_SERVICES_RECENT_PC', 'name_complete': False, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 32, 'name': 'RX_PAYER_MIX_DISCOUNT_CARD', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 33, 'name': 'VISIT_TYP_VST_TELEHEALTH', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 34, 'name': 'DX_SUBCAT_SPRAINS_AND_STRAINS_INITIAL_ENCOUNTER', 'name_complete': False, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 35, 'name': 'CPT_99203_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT', 'name_complete': False, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 36, 'name': 'NUM_HYPERSOMNIA_CLAIMS_L12M', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 37, 'name': 'VA_CLASS1_CN309_SEDATIVES_HYPNOTICS_OTHER', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 38, 'name': 'CPT_87880_DETECTION_TEST_BY_IMMUNOASSAY_WITH_DI', 'name_complete': False, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 39, 'name': 'CPT_90791_PSYCHIATRIC_DIAGNOSTIC_EVALUATION', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 40, 'name': '_99213_99214_G47411_', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 41, 'name': 'LEVEL_1_CPT_INFECTIOUS_AGENT_DETECTION_BY_DNA_R', 'name_complete': False, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 42, 'name': 'NUM_NEURO_SPECIALISTS', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 43, 'name': 'CPT_90471_ADMINISTRATION_OF_VACCINE', 'name_complete': True, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 44, 'name': '_G4733_G47411_', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 45, 'name': 'DX_SUBCAT_OTHER_SPECIFIED_UPPER_RESPIRATORY_INF', 'name_complete': False, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 46, 'name': 'ATC_1_SELECTIVE_SEROTONIN_REUPTAKE_INHIBITORS', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 47, 'name': 'CLAIMS_RX_REVERSED_RECENT_PCT', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'UNRESOLVED'},
    {'seq': 48, 'name': 'CPT_99395_ESTABLISHED_PATIENT_PERIODIC_PREVENTIV', 'name_complete': False, 'type_label': 'Numeric', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 49, 'name': 'LEVEL_2_CPT_MOLECULAR_TESTING', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
    {'seq': 50, 'name': 'DX_G4719_OTHER_HYPERSOMNIA', 'name_complete': True, 'type_label': 'Binary', 'default_status': 'SNAPSHOT_ONLY'},
]

# Review metadata is not a feature-selection list and cannot enable missing calculations.
# MODEL_TYPE/FINAL_MODEL still determine membership and tensor order.
SUPPLIED_FEATURE_REVIEW = {'_G47411_G47411_': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match G47411 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Snapshot replay omitted.', 'notes': 'Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match G47411 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, '_G4710_G47411_': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match G4710 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Snapshot replay omitted.', 'notes': 'Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match G4710 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, '_99213_99214_G47411_': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match 99213 followed by 99214 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Snapshot replay omitted.', 'notes': 'Draft: within rolling year ending PERIOD_END, combine medical diagnosis/procedure dates and paid pharmacy fill dates, insert a gap token for a >=90-day visit gap, retain last 10 visits, then match 99213 followed by 99214 followed by G47411. Draft decreasing visit offsets make the max-skip check ineffective; ordering/multiplicity and observation coverage need confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'NUM_SPECIALISTS': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.GATTEX_TRIGGER_AUTO.HCP_LOOKUP. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: count distinct rendering/prescriber NPIs in the rolling year ending PERIOD_END after current active HCP_LOOKUP maps them to sleep medicine, neurology, pulmonology or psychiatry. Historical specialty, coverage and source availability are unverified. Snapshot replay omitted.', 'notes': 'Draft: count distinct rendering/prescriber NPIs in the rolling year ending PERIOD_END after current active HCP_LOOKUP maps them to sleep medicine, neurology, pulmonology or psychiatry. Historical specialty, coverage and source availability are unverified. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'SLEEP_MED_VISIT_RECENCY_PCT': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.GATTEX_TRIGGER_AUTO.HCP_LOOKUP. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: distinct sleep-medicine event dates in PERIOD_END-90 days through PERIOD_END divided by distinct sleep-medicine event dates in the rolling year; zero when denominator is zero. Uses current active HCP_LOOKUP. Historical specialty, zero-denominator policy and coverage require confirmation. Snapshot replay omitted.', 'notes': 'Draft: distinct sleep-medicine event dates in PERIOD_END-90 days through PERIOD_END divided by distinct sleep-medicine event dates in the rolling year; zero when denominator is zero. Uses current active HCP_LOOKUP. Historical specialty, zero-denominator policy and coverage require confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'AVG_AT_RX_NOZOLP': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_TA_PRIVATE.TAK861.TAK861_TX_READY_V63_TMP_CUSTOM_HCP_ATOPEN_METRICS; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NOZOLP with zero, and average across joined claim rows. Persisted historical metrics may be unavailable; averaging is claim-row weighted, not one row per provider. The dictionary instead names AVG_AT_PTS_NOZOLP. Snapshot replay omitted.', 'notes': 'Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NOZOLP with zero, and average across joined claim rows. Persisted historical metrics may be unavailable; averaging is claim-row weighted, not one row per provider. The dictionary instead names AVG_AT_PTS_NOZOLP. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'MAX_AT_RX_NTILE_NOZOLP': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_TA_PRIVATE.TAK861.TAK861_TX_READY_V63_TMP_CUSTOM_HCP_ATOPEN_METRICS; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NTILE_NOZOLP with zero, and take maximum. Historical ntile construction/population and persisted period coverage are unverified; absent metric rows cannot establish zero activity. Snapshot replay omitted.', 'notes': 'Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NTILE_NOZOLP with zero, and take maximum. Historical ntile construction/population and persisted period coverage are unverified; absent metric rows cannot establish zero activity. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'MAX_AT_RX_NTILE_WITHZOLP': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_TA_PRIVATE.TAK861.TAK861_TX_READY_V63_TMP_CUSTOM_HCP_ATOPEN_METRICS; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NTILE_WITHZOLP with zero, and take maximum. Historical ntile construction/population and persisted period coverage are unverified; absent metric rows cannot establish zero activity. Snapshot replay omitted.', 'notes': 'Draft: join rolling-year medical/pharmacy provider claim rows to ATOPEN_METRICS by NPI and matching START_DT/END_DT, replace null AT_RX_NTILE_WITHZOLP with zero, and take maximum. Historical ntile construction/population and persisted period coverage are unverified; absent metric rows cannot establish zero activity. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'TIMES_GENERIC_MIX_ADJUSTED': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.TAK861.NARCOLEPSY_MARKET_BASKET_CODES. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: join pharmacy NDC11 to market-basket CODE; keep supplied generic therapeutic groups. Build daily distinct GENERIC_NAME counts from PERIOD_END-270 days through PERIOD_END-1 day; count changes from the preceding day. Coverage uses FILL_DATE through FILL_DATE+DAYS_SUPPLY inclusive, and no paid-status filter is present. Effective-data-end anchor, days-supply boundary and observation coverage require confirmation. Snapshot replay omitted.', 'notes': 'Draft: join pharmacy NDC11 to market-basket CODE; keep supplied generic therapeutic groups. Build daily distinct GENERIC_NAME counts from PERIOD_END-270 days through PERIOD_END-1 day; count changes from the preceding day. Coverage uses FILL_DATE through FILL_DATE+DAYS_SUPPLY inclusive, and no paid-status filter is present. Effective-data-end anchor, days-supply boundary and observation coverage require confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'UNIQUE_GENERICS_TRIED': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.TAK861.NARCOLEPSY_MARKET_BASKET_CODES. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: join pharmacy NDC11 to market-basket CODE; keep supplied generic therapeutic groups, then count distinct BRAND_NAME over all fills before PERIOD_END. No paid-status filter is present. BRAND_NAME versus GENERIC_NAME, history completeness and missing-name treatment require confirmation. Snapshot replay omitted.', 'notes': 'Draft: join pharmacy NDC11 to market-basket CODE; keep supplied generic therapeutic groups, then count distinct BRAND_NAME over all fills before PERIOD_END. No paid-status filter is present. BRAND_NAME versus GENERIC_NAME, history completeness and missing-name treatment require confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'NUM_DISCONTINUATIONS': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.TAK861.NARCOLEPSY_MARKET_BASKET_CODES. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: form daily distinct GENERIC_NAME counts in PERIOD_END-270 days through PERIOD_END-1 day; count downward transitions among change rows. Source joins NDC11 to market-basket CODE and supplied generic groups; there is no paid-status filter. Effective-data-end anchor, inclusive days-supply boundary, first transition and observation coverage require confirmation. Snapshot replay omitted.', 'notes': 'Draft: form daily distinct GENERIC_NAME counts in PERIOD_END-270 days through PERIOD_END-1 day; count downward transitions among change rows. Source joins NDC11 to market-basket CODE and supplied generic groups; there is no paid-status filter. Effective-data-end anchor, inclusive days-supply boundary, first transition and observation coverage require confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'NUM_GAPS_30_PLUS_DAYS': {'status': 'UNRESOLVED', 'source': 'Declared draft sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_TA_PRIVATE.TAK861.NARCOLEPSY_MARKET_BASKET_CODES. Current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Draft: form daily distinct GENERIC_NAME counts in PERIOD_END-270 days through PERIOD_END-1 day; count zero-count runs when their day index reaches exactly 30. Source joins NDC11 to market-basket CODE and supplied generic groups; there is no paid-status filter. This counts runs of at least 30 days, whereas the dictionary says over 30; anchor and coverage also need confirmation. Snapshot replay omitted.', 'notes': 'Draft: form daily distinct GENERIC_NAME counts in PERIOD_END-270 days through PERIOD_END-1 day; count zero-count runs when their day index reaches exactly 30. Source joins NDC11 to market-basket CODE and supplied generic groups; there is no paid-status filter. This counts runs of at least 30 days, whereas the dictionary says over 30; anchor and coverage also need confirmation. Event dates do not establish record availability or enrolled observation coverage; no fabricated provenance is accepted.', 'implementation_stage': 'DRAFT_EVENT_LOGIC_NOT_ENABLED'}, 'AGE': {'status': 'APPROXIMATED', 'source': 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PATIENT_DEMOGRAPHICS_LATEST; cohort restriction: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_DL_POC_SNAPSHOTS', 'logic': 'Candidate name requires runtime V63 reconciliation. Implemented: raw AGE = year(PERIOD_END) - YEAR_OF_BIRTH for cohort patients, followed only downstream by CEIL(raw AGE/10). Missing YOB stays unavailable. Birth year is treated as stable; historical availability of the current demographics extract is approximated. No event/availability dates are invented.', 'notes': 'Current demographic extract has no verified historical availability/correction history; missing YOB unavailable.', 'implementation_stage': 'IMPLEMENTED_AGE'}, 'CPT_95805_SLEEP_STUDY_MULTIPLE_TRIALS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims with CPT 95805 (sleep study) in rolling one-year window ending at PERIOD_END; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CPT_99204_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT_VISIT_WITH_MODERATE_LEVEL_OF_MEDICAL_DECISION_MAKING_IF_USING_TIME_45_MINUTES_OR_MORE': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims with CPT 99204 (new patient office visit, moderate MDM) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CPT_99203_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT_VISIT_WITH_LOW_LEVEL_OF_MEDICAL_DECISION_MAKING_IF_USING_TIME_30_MINUTES_OR_MORE': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims with CPT 99203 (new patient office visit, low MDM) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CPT_90791_PSYCHIATRIC_DIAGNOSTIC_EVALUATION': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims with CPT 90791 (psychiatric diagnostic evaluation) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CPT_87880_DETECTION_TEST_BY_IMMUNOASSAY_WITH_DIRECT_VISUAL_OBSERVATION_FOR_STREPTOCOCCUS_GROUP_A_STREP': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims with CPT 87880 (immunoassay strep test) in rolling one-year window ending at PERIOD_END; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'DX_G47411_NARCOLEPSY_WITH_CATAPLEXY': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of any ICD-10 G47.411 diagnosis claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'DX_CAT_DISEASES_OF_THE_NERVOUS_SYSTEM': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of any nervous system disease diagnosis (ICD-10 G-codes) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'DX_SUBCAT_OTHER_SPECIFIED_AND_UNSPECIFIED_UPPER_RESPIRATORY_DISEASE': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of upper respiratory disease diagnosis subcategory in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'PL_22': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of on-campus outpatient hospital visit (POS 22) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'VISIT_TYP_VST_TELEHEALTH': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of telehealth visit in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'PL_11_RECENT_PCT': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Recent percentage of POS 11 (office) visits; Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'L12M_NARCO_CLAIMS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count narcolepsy-related diagnosis claims (G47411,G47421,G47419,G47429,G4711,G4712) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'L6M_NARCO_CLAIMS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary presence of narcolepsy diagnosis claims in six-month window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'NARCO_CLAIMS_RECENT_RATIO': {'status': 'UNRESOLVED', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Ratio of narcolepsy claims in L3M (cutoff-90 to cutoff) versus lifetime; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder. Definition conflict: dictionary uses L3M/L12M; supplied metadata uses L3M/lifetime.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing. Denominator conflicts: L12M versus lifetime.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'NUM_EXCESSIVE_DAYTIME_SLEEPINESS_CLAIMS_L12M': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count excessive daytime sleepiness diagnosis claims (R400,G4710-G4714,G4719) in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'NUM_CATAPLEXY_CLAIMS_L3M': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count cataplexy diagnosis claims (G47411,G47421,G47431,G47419) in L3M window (cutoff-90 to cutoff); V63 Binary encoding downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'NUM_HYPERSOMNIA_CLAIMS_L3M': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count hypersomnia diagnosis claims (G4710-G4714,G4719,F511) in L3M window (cutoff-90 to cutoff). Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'NUM_PSYCH_COMORBIDITIES_L12M': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count psychiatric comorbidities (depression F32-F33, anxiety F40-F41, ADHD F90, bipolar F31) in rolling one-year window; V63 Binary encoding downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'VA_CLASS1_CN809_CNS_STIMULANTS_OTHER': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of CNS stimulant (VA class CN809) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'VA_CLASS1_CN801_AMPHETAMINES': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of amphetamine (VA class CN801) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_1_OTHER_ANTIDEPRESSANTS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of other antidepressant (ATC N06AX) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_2_HYPNOTICS_AND_SEDATIVES': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of hypnotic/sedative (ATC N05C) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_3_STOMATOLOGICAL_PREPARATIONS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of stomatological preparation (ATC A01) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_1_OTHER_ANTIPSYCHOTICS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of other antipsychotic (ATC N05AX) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_3_PSYCHOLEPTICS': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of psycholeptic (ATC N05) pharmacy claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'LEVEL_1_CPT_IMMUNOASSAY': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of immunoassay CPT category claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'LEVEL_1_CPT_INFECTIOUS_AGENT_DETECTION_BY_DNA_RNA': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims in infectious agent DNA/RNA detection CPT category in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'LEVEL_1_CPT_SLEEP_STUDY': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count medical claims in sleep study CPT category in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'LEVEL_1_CPT_OFFICE_E_M_NEW': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of new-patient office E/M CPT category claim in rolling one-year window ending at PERIOD_END. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'RX_INSURANCE_SEGMENT_INTEGRATED': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST; DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PLANS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of pharmacy claim paid by integrated insurance segment in L12M window. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'RX_PAYER_MIX_DISCOUNT_CARD': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of pharmacy claim with discount card payer mix in L12M window. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'RX_PAYER_MIX_COMMERCIAL_RECENT_PCT': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Recent percentage of commercial payer mix claims; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CLAIMS_RX_REJECTED': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Count/binary flag of rejected pharmacy claims in rolling one-year window; V63 Binary encoding downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'CLAIMS_RX_REVERSED_RECENT_PCT': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Recent percentage of reversed pharmacy claims; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'ATC_3_PSYCHOANALEPTICS_RECENT_PCT': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST. Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Recent percentage of psychoanaleptic (ATC N06) pharmacy claims; V63 Binary encoding applied downstream. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}, 'HCPS_PX_SP_PRIMARY_NURSE_PRACTITIONER': {'status': 'SNAPSHOT_ONLY', 'source': 'Declared upstream sources: DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST; provider specialty lookup (table not specified). Supplied builder reads only current snapshot: DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_MODEL_DATA', 'logic': 'Candidate name requires runtime V63 reconciliation. Not enabled. Historical calculation not supplied; snapshot replay omitted. Declared definition only: Binary flag for presence of medical claim with primary nurse practitioner provider type in L12M window. Exact code mappings, event grain, status/date filters and observation coverage are not implemented in the supplied builder.', 'notes': 'Only snapshot replay is supplied; historical executable calculation, source availability and observation coverage are missing.', 'implementation_stage': 'SNAPSHOT_REPLAY_OMITTED'}}

HISTORICAL_RULES = {}
for name in features:
    if name in SUPPLIED_FEATURE_REVIEW and name != 'AGE':
        item = SUPPLIED_FEATURE_REVIEW[name]
        HISTORICAL_RULES[name] = {key: item[key] for key in ('status', 'source', 'logic', 'notes')}
HISTORICAL_RULES.update(register_age_rules(features, read_query, DATABASE, PREFIX))


In [ ]:
# Compare supplied names with the configured model without adding or removing predictors.
supplied_name_audit = supplied_feature_reconciliation(features, SUPPLIED_FEATURE_REVIEW)
print('Supplied names are candidates; authoritative V63 membership/order remains unchanged.')
display(supplied_name_audit)
display(supplied_name_audit.NAME_MATCH.value_counts().rename_axis('NAME_MATCH').reset_index(name='COUNT'))
print('Supplied calculation evidence (these are not model-performance results):')
display(pd.DataFrame([dict(FEATURE_NAME=name, STATUS=item['status'],
    IMPLEMENTATION_STAGE=item['implementation_stage'], SOURCE=item['source'],
    CALCULATION=item['logic'], GAPS=item['notes'])
    for name, item in SUPPLIED_FEATURE_REVIEW.items()]))
supplied_dictionary_audit, supplied_dictionary_only = reconcile_dictionary(
    list(SUPPLIED_FEATURE_REVIEW), BUSINESS_DICTIONARY_REFERENCE)
print('Supplied names versus the earlier 50-entry dictionary:')
display(supplied_dictionary_audit[['FEATURE_NAME', 'DICTIONARY_SEQ', 'MATCH']])
display(supplied_dictionary_only)
print('Matched:', int(supplied_dictionary_audit.MATCH.ne('UNRESOLVED').sum()),
      '| Unmatched supplied names:', int(supplied_dictionary_audit.MATCH.eq('UNRESOLVED').sum()),
      '| Dictionary-only entries:', len(supplied_dictionary_only))
print('Different names are not aliases. Counts alone cannot resolve the earlier 49-versus-50 discrepancy.')

# Display dictionary reconciliation and all 49 reconstruction statuses
audit, status_counts, dictionary_only = reconstruction_audit(features, BUSINESS_DICTIONARY_REFERENCE, HISTORICAL_RULES, business_parameters)
display(audit)
display(status_counts.rename_axis('STATUS').reset_index(name='FEATURE_COUNT'))
for status, count in status_counts.items():
    print(f'{status} = {count}')
print('Status total:', int(status_counts.sum()))
print('Dictionary reference rows:', len(BUSINESS_DICTIONARY_REFERENCE), '| Authoritative model inputs:', len(features))
if not dictionary_only.empty:
    print('Dictionary rows not matched to the authoritative model list (clipped text may require confirmation):')
    display(dictionary_only)
if len(dictionary_only) == 1 and audit.MATCH.ne('UNRESOLVED').all():
    print('The reference dictionary has one entry outside the configured predictor list:', dictionary_only.iloc[0]['name'])
    print('It is not added as a 50th feature. Why it was excluded upstream is not inferred here.')
else:
    print('Do not infer the 49-versus-50 cause until unmatched/clipped names are resolved against V63.')


In [ ]:
# Construct and inspect date grids before any historical value calculation
monthly_grid = sequence_grid(metadata, 'MONTHLY')
quarterly_grid = sequence_grid(metadata, 'QUARTERLY')
verify_periods(monthly_grid, quarterly_grid)
print('TIME_STEP 0 = most recent; increasing indices = older. Current bucket ends at END_DT.')
print('Date-grid rows only; these are not reconstructed model inputs:', len(monthly_grid), len(quarterly_grid))
display(monthly_grid.head(24).rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'}))
display(quarterly_grid.head(8).rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'}))
print('Supplied rolling-date rules shown at monthly and quarterly cutoffs; these are not patient-coverage evidence:')
display(documented_feature_windows(monthly_grid.head(12)))
display(documented_feature_windows(quarterly_grid.head(4)))
print('Supplied source references and discoverable schemas; unresolved columns/formulas must be confirmed before reconstruction:')
display(pd.DataFrame(SOURCE_REGISTRY, columns=['SOURCE', 'TABLE', 'EVENT_DATE_COLUMN', 'NOTES']))
display(inspect_source_schemas())


In [ ]:
# Stop on unsupported reconstruction; build monthly first and quarterly second
require_reconstruction(audit, HISTORICAL_RULES)
bundles = {}
bundles['MONTHLY'] = construct_sequence(monthly_grid, features, audit, HISTORICAL_RULES, 'MONTHLY', encoding_contract)
bundles['QUARTERLY'] = construct_sequence(quarterly_grid, features, audit, HISTORICAL_RULES, 'QUARTERLY', encoding_contract)
verify_periods(bundles['MONTHLY']['long'], bundles['QUARTERLY']['long'])
print('Actual monthly tensor shape:', bundles['MONTHLY']['X'].shape)
print('Actual quarterly tensor shape:', bundles['QUARTERLY']['X'].shape)


In [ ]:
# Inspect values, padding, sparsity and patient histories
coverage_rows = []
for name in ('MONTHLY', 'QUARTERLY'):
    bundle = bundles[name]
    report, feature_sparsity, history_distribution = sparsity_report(bundle, features)
    coverage_rows.append(report)
    print(name, 'feature sparsity on observed feature values; padded zeros excluded from that denominator')
    display(feature_sparsity)
    display(history_distribution)
    display_sequence(bundle, features)
    # Model padding demonstration uses the same patient/date/step keys as the raw display.
    model_sample = bundle['long'][['PATIENT_ID', 'END_DT', 'RESP', 'TIME_STEP', 'IS_VALID_TIMESTEP', 'IS_PADDED']].head(24).copy()
    model_sample[features] = bundle['X'].reshape(-1, 49)[:len(model_sample)]
    print(name, 'V63 encoded and padded tensor values before TRAIN-only standardization')
    display(model_sample)
display(pd.DataFrame(coverage_rows))
print('Feature sparsity is not timestep padding; numeric zero after scaling is not a sparsity measure.')


In [ ]:
# Save verified temporal inputs privately for the same four-notebook pipeline
artifacts = prepared_blobs(metadata, features, bundles, audit, configuration_audit, snapshot_X, encoding_contract)
save_artifacts(PREPARED_TABLE, artifacts)
print('Prepared inputs saved and verified:', PREPARED_TABLE)
print('Continue with notebook 02. No patient records, tensors or outputs belong in the repository.')
